# How fast wildfires move (NIROPS EDA)

Problem hooks for the video, pitch and Figma deck: how fast big Western US wildfires grow, from nightly infrared fire maps (737 fires, 2020 to 2024).

**The data never goes in git.** It lives in `analysis/data/`, which is gitignored. The notebook looks for the Mendeley zip on your machine first: the `NIROPS_ZIP` env var, then `analysis/data/`, then `~/Downloads`. If it finds nothing, it downloads only the 1.4 GB shapefile zip from Mendeley. The first run builds a small cache (a few minutes). After that, Run All takes seconds.

**Setup:** `pip install geopandas pyogrio pyarrow matplotlib pillow`. ffmpeg is optional: with it you get MP4 time-lapses, without it GIFs. Images land in `analysis/outputs/` (1920x1080, dark, ready for the video).

**Credit on screen (CC BY 4.0):** Magstadt, O'Connor et al., "A high spatial resolution daily fire perimeter progression dataset for wildfires in the Western United States: 2020-2024", Mendeley Data, doi:10.17632/95rj5d379g.1

In [ ]:
import json, os, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "fire_spread_eda.py").exists() else Path.cwd() / "analysis"
sys.path.insert(0, str(HERE))
import geopandas as gpd
import pandas as pd
from IPython.display import Image, display
import fire_spread_eda as fse

STATE = "CA"     # headline scope; None = all Western US
FIRE = None      # e.g. "dixie" to choose the time-lapse fire; None = biggest fire in STATE
NEAR_KM = 100    # radius for "fires near Berkeley"
FPS = 3          # time-lapse speed, maps per second

OUT = fse.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Get the data (your local zip first, Mendeley if missing)

In [ ]:
if fse.CACHE.exists():
    print(f"Cache found: {fse.CACHE} ({fse.CACHE.stat().st_size / 1e6:.0f} MB). Delete it to rebuild.")
else:
    zp = fse.find_data()
    if zp is None:
        print("No local zip found (set NIROPS_ZIP=/path/to/your.zip to use one). Downloading instead.")
        zp = fse.download_vector()
    print("Using", zp)
    vector_zip = fse.ensure_vector_zip(zp)      # pulls only the shapefile zip out of the full download
    sources = fse.vector_sources(vector_zip)    # read in place, nothing else unpacked or run
    print("Reading", *sources, sep="\n  ")
    fse.build_cache(sources)

## 2. Load and clean

In [ ]:
gdf = gpd.read_parquet(fse.CACHE)
attrs = pd.DataFrame(gdf.drop(columns=gdf.geometry.name))
cols = fse.detect_columns(attrs)
work, how = fse.build_work(attrs, cols)
n = fse.nightly_growth(work)
print(f"{len(gdf):,} perimeters, {work['inc'].nunique():,} fires, "
      f"{work['t'].min():%b %Y} to {work['t'].max():%b %Y}. Timestamps from {how}.")
print("Columns used:", {k: v for k, v in cols.items() if v})
display(work["state"].value_counts().rename("perimeters").to_frame().T)
attrs.head(3)

## 3. Headline numbers

A "fire-day" is one fire between two consecutive nightly maps (12 to 36 hours apart).

In [ ]:
scoped = n[n["state"] == STATE] if STATE else n
h = fse.headline(scoped, STATE or "all")
h_all = fse.headline(n, "all")
summary = {"dataset": fse.CREDIT, "timestamps_from": how, "headline": h, "western_us": h_all}
flat = lambda x: {k: v for k, v in x.items() if not isinstance(v, (dict, list))}
display(pd.DataFrame([flat(h), flat(h_all)]))
pd.DataFrame(h["top10"])

## 4. Video cards and the top-10 chart

In [ ]:
paths = [fse.chart_top10(h, OUT / "top10_one_day_jumps.png"), *fse.hero_cards(h, OUT)]
for p in paths:
    display(Image(filename=str(p), width=960))

## 5. One fire, map by map (time-lapse)

In [ ]:
def timelapse(inc, tag=""):
    meta = n[n["inc"] == inc].iloc[0]
    name, st = fse.pretty(meta["name"]), meta["state"]
    rows = work.index[work["inc"] == inc]
    frames = fse.frames_from_gdf(gdf.loc[rows].assign(night=work.loc[rows, "night"]))
    acres = n[n["inc"] == inc].set_index("night")["acres"]
    stem = f"timelapse_{tag}{fse.slug(name)}"
    anim = fse.chart_timelapse(frames, name, st, acres, OUT / f"{stem}.png", OUT / f"{stem}.mp4", FPS)
    display(Image(filename=str(OUT / f"{stem}.png"), width=960))
    print("Animation saved:", anim)

pool = n[n["name"].str.contains(FIRE, case=False, na=False)] if FIRE else scoped
timelapse(pool.groupby("inc")["acres"].max().idxmax())

## 6. Fires near Berkeley

In [ ]:
from shapely.geometry import Point

berkeley = gpd.GeoSeries([Point(*fse.BERKELEY_LONLAT)], crs=4326).to_crs(5070).iloc[0]
km = gdf.loc[work.index].distance(berkeley) / 1000
near = pd.Series(km.values, index=work["inc"].values).groupby(level=0).min()
near = near[near <= NEAR_KM].rename("closest_km_to_berkeley")
final = n.groupby("inc").agg(fire=("name", "first"), state=("state", "first"),
                             first_map=("night", "min"), final_acres=("acres", "max"))
near_tbl = final.join(near, how="inner").sort_values("final_acres", ascending=False)
near_tbl["fire"] = near_tbl["fire"].map(fse.pretty)
display(near_tbl.head(15))
if len(near_tbl):
    timelapse(near_tbl.index[0], tag="bay_area_")
else:
    print(f"No mapped fires within {NEAR_KM} km of Berkeley.")

## 7. How far the fire edge moved in one day

The farthest new burn from the previous map's edge. Only growth attached to the old footprint counts, so a new spot fire miles away can't inflate it.

In [ ]:
one = scoped[scoped["one_night"]]
best = []
for inc in one.sort_values("growth", ascending=False).drop_duplicates("inc").head(15)["inc"]:
    rows = work.index[work["inc"] == inc]
    frames = fse.frames_from_gdf(gdf.loc[rows].assign(night=work.loc[rows, "night"]))
    best += [{"inc": inc, **r} for r in fse.front_runs(frames) if r["gap_days"] == 1]
runs = pd.DataFrame(best)
if runs.empty:
    print("No one-day front runs found.")
else:
    runs = (runs.join(n.set_index(["inc", "night"])[["name", "state", "hours"]], on=["inc", "night"])
                .sort_values("front_km", ascending=False))
    runs["name"] = runs["name"].map(fse.pretty)
    display(runs.head(10))
    r = runs.iloc[0]
    card = fse.hero(f"{r['front_km']:.0f} km", f"the fire edge moved in {r['hours']:.0f} hours",
                    f"{r['name']} Fire ({r['state']}), {r['night']:%b %d, %Y}: farthest new burn "
                    "from the previous map's edge", OUT / "hero_front_run.png",
                    "Counts only growth attached to the old footprint (no spot fires).")
    display(Image(filename=str(card), width=960))
    summary["front_run"] = {"fire": r["name"], "state": r["state"], "night_of": str(r["night"].date()),
                            "front_km": float(r["front_km"]), "hours": float(r["hours"])}

## 8. Paste-ready lines

For `problem_stats.md`, the video script and the pitch. Every number below comes from the cells above.

In [ ]:
b, f = h["biggest_one_day_jump"], h["fastest_spread"]
where = "California" if STATE == "CA" else "the Western US"
lines = [
    f"In {where}, 2020-2024, {h['share_days_bigger_than_oakland_1991_pct']}% of mapped fire-days saw one fire "
    f"grow more than the entire 1991 Oakland Hills fire ({fse.OAKLAND_1991_ACRES:,} acres).",
    f"Biggest one-day jump: the {b['fire']} Fire added {b['acres_added']:,} acres in {b['hours_between_maps']:.0f} "
    f"hours, {b['x_oakland_1991']:.0f}x the Oakland Hills fire.",
    f"Fastest: the {f['fire']} Fire burned about {f['football_fields_per_min']:.0f} football fields a minute "
    f"for {f['hours_between_maps']:.0f} hours.",
]
if "front_run" in summary:
    fr = summary["front_run"]
    lines.append(f"The {fr['fire']} Fire's edge moved {fr['front_km']:.0f} km in {fr['hours']:.0f} hours.")
if len(near_tbl):
    t = near_tbl.iloc[0]
    lines.append(f"{len(near_tbl)} mapped fire{'s' if len(near_tbl) != 1 else ''} burned within {NEAR_KM} km of Berkeley since 2020; the biggest, "
                 f"{t['fire']} ({t['first_map']:%Y}), reached {t['final_acres']:,.0f} acres.")
lines.append("Source: NIROPS daily fire perimeters (Magstadt, O'Connor et al., Mendeley Data, "
             "doi:10.17632/95rj5d379g.1, CC BY 4.0).")
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("\n".join("- " + line for line in lines))